# 02 — Preparación de Datos

**TFM RunnAing** | Fases 3.1 · 3.2 · 3.3

**Objetivos:**
- 3.1 Limpieza: descartar sesiones con condiciones fisiológicas inválidas
- 3.2 Ingeniería de características: 13 features por sesión (GPS + cardíacas + HRV)
- 3.3 Cálculo del target TRIMP incremental de Banister (1991)

**Supuesto documentado**: FitRec no incluye FC_reposo ni FC_max por usuario.  
Se usan valores poblacionales: `HR_rest = 60 bpm`, `HR_max = 185 bpm` (conservador para adultos recreativos).

**Output**: `data/processed/sessions_features.parquet`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from src.data_loader import auto_detect_and_load, filter_running_sessions
from src.features import build_feature_matrix, FEATURE_NAMES
from src.trimp import banister_trimp_incremental, DEFAULT_HR_REST, DEFAULT_HR_MAX

np.random.seed(42)
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_EDA = Path('../reports/eda')
REPORTS_EDA.mkdir(parents=True, exist_ok=True)

print(f'HR_rest asumido : {DEFAULT_HR_REST} bpm  (FitRec no incluye FC_reposo por usuario)')
print(f'HR_max asumido  : {DEFAULT_HR_MAX} bpm  (FitRec no incluye FC_max por usuario)')
print('Setup OK')

## 1. Carga inicial

In [ ]:
df_raw = auto_detect_and_load('../data/raw')
df = filter_running_sessions(df_raw)

funnel = []
funnel.append({'paso': '0. Dataset running original',
               'n_sesiones': len(df), 'n_usuarios': df['userId'].nunique(), 'eliminadas': 0})
print(f"Paso 0 — running: {len(df):,} sesiones | {df['userId'].nunique():,} usuarios")

## 2. Limpieza (Fase 3.1)

Criterios de descarte (cualquiera de los siguientes):
1. Sin género
2. Sin FC o < 2 muestras
3. Sin GPS completo
4. **Duración < 10 minutos**
5. **FC media fuera de [30, 220] lpm**
6. **Altitud nula o ausente**
7. **Velocidad media > 25 km/h**

In [ ]:
# Paso 1 — género válido
n_prev = len(df)
df = df[df['gender'].notna()].copy()
funnel.append({'paso': '1. Género válido', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 1 género    : {len(df):,}  (-{n_prev - len(df):,})")

# Paso 2 — FC disponible
n_prev = len(df)
df = df[df['heart_rate'].apply(lambda x: isinstance(x, list) and len(x) >= 2)].copy()
funnel.append({'paso': '2. FC disponible (≥2 muestras)', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 2 FC        : {len(df):,}  (-{n_prev - len(df):,})")

# Paso 3 — GPS completo
n_prev = len(df)
df = df[
    df['latitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2) &
    df['longitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2) &
    df['altitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2) &
    df['timestamp'].apply(lambda x: isinstance(x, list) and len(x) >= 2)
].copy()
funnel.append({'paso': '3. GPS completo', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 3 GPS       : {len(df):,}  (-{n_prev - len(df):,})")

In [ ]:
def get_duration_min(ts_list):
    """Duración en minutos desde lista de timestamps."""
    try:
        ts = np.array(ts_list, dtype=float)
        if ts[-1] - ts[0] > 1e8:
            ts = ts / 1000.0
        return (ts[-1] - ts[0]) / 60.0
    except Exception:
        return 0.0

df['_dur'] = df['timestamp'].apply(get_duration_min)

# Paso 4 — Duración >= 10 min
n_prev = len(df)
df = df[df['_dur'] >= 10.0].copy()
funnel.append({'paso': '4. Duración ≥ 10 min', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 4 duración  : {len(df):,}  (-{n_prev - len(df):,})")

In [ ]:
df['_hr_mean'] = df['heart_rate'].apply(
    lambda x: float(np.nanmean([v for v in x if v > 0]))
    if isinstance(x, list) and len(x) > 0 else np.nan
)

# Paso 5 — FC media en [30, 220]
n_prev = len(df)
df = df[df['_hr_mean'].between(30, 220)].copy()
funnel.append({'paso': '5. FC media ∈ [30, 220] lpm', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 5 FC [30-220]: {len(df):,}  (-{n_prev - len(df):,})")

In [ ]:
df['_alt_mean'] = df['altitude'].apply(
    lambda x: float(np.nanmean([v for v in x if not np.isnan(float(v))]
                               if isinstance(x, list) else [])) 
    if isinstance(x, list) else np.nan
)

# Paso 6 — altitud no nula
n_prev = len(df)
df = df[df['_alt_mean'].notna() & (df['_alt_mean'].abs() > 0)].copy()
funnel.append({'paso': '6. Altitud no nula', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 6 altitud   : {len(df):,}  (-{n_prev - len(df):,})")

In [ ]:
def get_speed_mean_kmh(row):
    """Velocidad media (km/h) desde GPS y timestamps."""
    try:
        lat = np.radians(np.array(row['latitude'], dtype=float))
        lon = np.radians(np.array(row['longitude'], dtype=float))
        ts = np.array(row['timestamp'], dtype=float)
        if ts[-1] - ts[0] > 1e8:
            ts = ts / 1000.0
        dlat, dlon = np.diff(lat), np.diff(lon)
        a = np.sin(dlat/2)**2 + np.cos(lat[:-1])*np.cos(lat[1:])*np.sin(dlon/2)**2
        dist_km = float(np.sum(6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))))
        dur_h = (ts[-1] - ts[0]) / 3600.0
        return dist_km / dur_h if dur_h > 0 else np.nan
    except Exception:
        return np.nan

df['_speed'] = df.apply(get_speed_mean_kmh, axis=1)

# Paso 7 — velocidad media <= 25 km/h
n_prev = len(df)
df = df[df['_speed'].notna() & (df['_speed'] <= 25.0)].copy()
funnel.append({'paso': '7. Velocidad media ≤ 25 km/h', 'n_sesiones': len(df),
               'n_usuarios': df['userId'].nunique(), 'eliminadas': n_prev - len(df)})
print(f"Paso 7 velocidad : {len(df):,}  (-{n_prev - len(df):,})")

# Guardar funnel
funnel_df = pd.DataFrame(funnel)
print('\n=== FUNNEL DE LIMPIEZA ===')
print(funnel_df.to_string(index=False))
funnel_df.to_csv(REPORTS_EDA / 'funnel_limpieza.csv', index=False)

# Limpiar auxiliares
df = df.drop(columns=['_dur', '_hr_mean', '_alt_mean', '_speed'], errors='ignore')

## 3. Ingeniería de características (Fase 3.2)

**13 features** por sesión:

| # | Feature | Descripción |
|---|---------|-------------|
| 1 | duration_min | Duración total (min) |
| 2 | distance_km | Distancia total (km) — Haversine |
| 3 | speed_mean | Velocidad media (km/h) |
| 4 | speed_max | Velocidad máxima (km/h) |
| 5 | speed_std | Desviación estándar velocidad |
| 6 | pace_mean | Ritmo medio (min/km) |
| 7 | elevation_gain | Desnivel positivo acumulado (m) |
| 8 | altitude_mean | Altitud media (m) |
| 9 | grade_factor | elevation_gain / distance_km |
| 10 | hr_mean | FC media (bpm) |
| 11 | hr_max | FC máxima (bpm) |
| 12 | hr_min | FC mínima (bpm) |
| 13 | hrv_estimate | HRV = std(RR_intervals), RR=60/FC_inst (Shcherbina et al., 2017) |

In [ ]:
print(f'Calculando 13 features para {len(df):,} sesiones...')
X = build_feature_matrix(df, include_hr=True)
print(f'\nFeatures: {X.shape}  — columnas: {list(X.columns)}')
print(f'\nNaN por columna:')
print(X.isna().sum().to_string())

In [ ]:
n_before = len(X)
valid_mask = X.notna().all(axis=1)
X = X[valid_mask]
df_clean = df[valid_mask].copy()
print(f'Sesiones con features completas: {len(X):,}  (descartadas: {n_before - len(X):,})')
print()
print('Estadísticos de features:')
print(X.describe([0.25, 0.5, 0.75]).round(2).to_string())

## 4. Target TRIMP incremental (Fase 3.3)

```
TRIMP = Σ_t ( Δt · HRR_t · e^(b · HRR_t) )
HRR_t = (FC_inst_t − HR_rest) / (HR_max − HR_rest)
b = 1.92 (hombres) | 1.67 (mujeres)
```

In [ ]:
print(f'Calculando TRIMP incremental para {len(df_clean):,} sesiones...')
trimp_values = [
    banister_trimp_incremental(
        heart_rate=row['heart_rate'],
        timestamps=row['timestamp'],
        gender=row['gender'],
        hr_rest=DEFAULT_HR_REST,
        hr_max=DEFAULT_HR_MAX,
    )
    for _, row in df_clean.iterrows()
]
y = pd.Series(trimp_values, index=df_clean.index, name='trimp')

valid_trimp = y.notna() & (y > 0)
y = y[valid_trimp]
X = X[valid_trimp]
df_clean = df_clean[valid_trimp]

print(f'Sesiones con TRIMP válido: {len(y):,}')
print('\nTRIMP — estadísticos:')
print(y.describe([0.25, 0.5, 0.75, 0.90, 0.95]).round(2).to_string())

## 5. Guardado del dataset procesado

In [ ]:
def extract_session_date(ts_list):
    """Extrae la fecha de inicio de sesión desde la lista de timestamps."""
    try:
        ts = min(ts_list)
        if ts > 1e9:
            ts /= 1000.0
        return pd.to_datetime(ts, unit='s').normalize()
    except Exception:
        return pd.NaT

df_final = df_clean[['userId', 'gender']].copy()
df_final['date'] = df_clean['timestamp'].apply(extract_session_date)
df_final = pd.concat([df_final, X, y], axis=1)

assert 'trimp' in df_final.columns
assert df_final['trimp'].notna().all()
assert df_final[list(X.columns)].notna().all().all()

out_path = PROCESSED / 'sessions_features.parquet'
df_final.to_parquet(out_path, index=True)

print(f'Guardado: {out_path}')
print(f'Shape   : {df_final.shape}')
print(f'Columnas: {list(df_final.columns)}')

In [ ]:
print('=== RESUMEN FASE 3 ===')
orig = funnel[0]['n_sesiones']
print(f'Sesiones running originales : {orig:,}')
print(f'Sesiones dataset final      : {len(df_final):,}')
print(f'Reducción total             : {100*(1 - len(df_final)/orig):.1f}%')
print(f'Usuarios únicos             : {df_final["userId"].nunique():,}')
print(f'Features por sesión         : {len(X.columns)}')
print(f'TRIMP medio                 : {y.mean():.2f}')
print(f'TRIMP mediana               : {y.median():.2f}')
print(f'\nSupuestos:')
print(f'  HR_rest = {DEFAULT_HR_REST} bpm (FitRec no incluye FC_reposo por usuario)')
print(f'  HR_max  = {DEFAULT_HR_MAX} bpm (FitRec no incluye FC_max por usuario)')